In [ ]:
import gymnasium as gym
import numpy as np

## Value iterator

In [ ]:
def value_iteration(entorno, gamma=0.99, theta=1e-8):
    """
    Implementa el algoritmo Value Iteration (Iteración de Valor).

    Fundamento teórico:
    -------------------
    Aplica la ecuación de Bellman iterativamente sobre todos los estados
    hasta convergencia. En cada barrida actualiza:

        V(s) = max_a { Σ P(s'|s,a) · [R + γ · V(s')] }

    Al converger, extrae la política óptima:
        π*(s) = argmax_a Q(s,a)

    Parámetros:
    -----------
    entorno : gym.Env  — entorno Gym con acceso a entorno.P (modelo del entorno)
    gamma   : float    — factor de descuento (0 < γ ≤ 1)
    theta   : float    — criterio de convergencia (umbral de cambio mínimo)

    Retorna:
    --------
    V       : np.ndarray — función de valor óptima para cada estado
    politica: np.ndarray — política óptima (acción por estado)
    iteraciones: int     — cantidad de iteraciones hasta convergencia
    """

    # ── Paso 1: Obtener dimensiones del espacio de estados y acciones ──────
    num_estados = entorno.observation_space.n
    num_acciones = entorno.action_space.n

    # ── Paso 2: Inicializar la función de valor en 0 para todos los estados ─
    V = np.zeros(num_estados)  # V[s] = valor del estado s

    # Acceder a la dinámica del entorno (env.unwrapped.P en Gymnasium moderno)
    # P[estado][accion] = lista de (probabilidad, sig_estado, recompensa, terminal)
    modelo = entorno.unwrapped.P

    iteraciones = 0  # Contador de barridas (sweeps)

    # ── Paso 3: Bucle principal hasta convergencia ─────────────────────────
    while True:
        delta = 0.0  # Máximo cambio en esta barrida (criterio de parada)
        iteraciones += 1

        # Recorrer todos los estados
        for estado in range(num_estados):
            valor_anterior = V[estado]  # Guardar valor previo para calcular Δ

            # Calcular el valor Q para cada acción posible en este estado
            # modelo[estado][accion] = lista de (prob, sig_estado, recompensa, terminal)
            valores_q = np.zeros(num_acciones)
            for accion in range(num_acciones):
                for prob_transicion, sig_estado, recompensa, es_terminal in modelo[
                    estado
                ][accion]:
                    # Ecuación de Bellman: suma sobre estados siguientes ponderada por probabilidad
                    # En modo determinístico: prob_transicion = 1.0 siempre
                    # En modo estocástico:   prob_transicion < 1.0 (el entorno puede "resbalar")
                    valores_q[accion] += prob_transicion * (
                        recompensa + gamma * V[sig_estado]
                    )

            # Actualizar V(s) con el máximo Q-valor (acción óptima)
            V[estado] = np.max(valores_q)

            # Actualizar el cambio máximo de esta barrida
            delta = max(delta, abs(valor_anterior - V[estado]))

        # ── Verificar convergencia: si Δ < θ, detener ─────────────────────
        if delta < theta:
            break

    # ── Paso 4: Extraer la política óptima a partir de V ──────────────────
    politica = np.zeros(num_estados, dtype=int)
    for estado in range(num_estados):
        # Q(s,a) = Σ P(s'|s,a) · [R + γ · V(s')]
        valores_q = np.zeros(num_acciones)
        for accion in range(num_acciones):
            for prob_transicion, sig_estado, recompensa, es_terminal in modelo[estado][
                accion
            ]:
                valores_q[accion] += prob_transicion * (
                    recompensa + gamma * V[sig_estado]
                )

        # La política elige la acción que maximiza Q(s,a)
        politica[estado] = np.argmax(valores_q)
